# 01 · Environment setup, GPU check and dataset access test

**Purpose.** Verify the GPU, CUDA and libraries, then test (I-1) whether the CircuitNet folder is reachable
through *your* Drive mount and check free storage (I-2). Nothing is downloaded or trained here.

**Before running:** Runtime → Change runtime type → **A100 GPU**. Upload `verihgn_code.zip` to
`MyDrive/VeriHGN/code/`. For the Drive route, open the
[CircuitNet folder](https://drive.google.com/drive/folders/1GjW-1LBx1563bg3pHQGvhcEyK2A9sYUB) and choose
*Organize → Add shortcut → My Drive*.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

## 1. GPU, CUDA, library versions

In [ ]:
import json, subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
from verihgn_repro.utils import hardware
from verihgn_repro import official
env = hardware.env_report({'scatter_impl': official.scatter_impl(), 'official_commit': official.official_commit()})
print(json.dumps(env, indent=1))
if env['hardware']['device'] != 'cuda':
    print('⚠ No GPU: select Runtime → Change runtime type → A100. CPU is only usable for smoke tests.')

## 2. Dataset access test (I-1) and storage check (I-2)

In [ ]:
from verihgn_repro.colab import access_check, storage_report, write_json
acc = access_check('/content/drive')
print(json.dumps(acc, indent=1))
st = storage_report(L)
print(json.dumps(st, indent=1))
write_json({'env': env, 'access': acc, 'storage': st}, L.reports / '01_environment.json')
if acc['ok']:
    print('✔ Drive route works: archives found via', acc['found']['method'])
else:
    print('✘ Drive route not available yet.', acc.get('instructions', ''))

## 3. Public-link fallback test (only if the Drive route failed)
Downloads the 124 MB `graph_infromation.tar.gz` with `gdown` and checks its byte size.

In [ ]:
TEST_GDOWN = not acc['ok']          # set True to test it anyway
if TEST_GDOWN:
    import time
    from verihgn_repro.data import circuitnet as cn
    t0 = time.time()
    p = cn.download_with_gdown(cn.ARCHIVES['graph_information'], L.local_archives)
    print('✔ gdown ok:', p, f'{time.time()-t0:.0f}s')

**Decide I-2 now.** Persistent data this stage writes to Drive: processed samples ≈ 3–4 GB (estimated from
real samples: 2–14 MB each), reports, and benchmark checkpoints (a few MB each). Caching the raw archives too
would add ≈ 12.4 GB (graph info 0.12 + gcell 1.96 + micron 2.72 + routability 7.59). Set
`CACHE_ON_DRIVE = True` in notebook 02 only if the free Drive space above comfortably exceeds ~17 GB.